In [30]:
import numpy as np
import pandas as pd
from pyexpat import features

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix
from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine

In [31]:
def append_unified_metric(dataframe: pd.DataFrame,
                          acc_col: str = 'avg_Accuracy_Seed_Mean',
                          f1_col: str = 'avg_F1-Score_Seed_Mean',
                          target_col: str = 'Acc_F1_Seed_Mean') -> pd.DataFrame:

    df_transformed = dataframe.copy()
    df_transformed[target_col] = (df_transformed[acc_col] + df_transformed[f1_col]) / 2.0
    return df_transformed

def extract_optimal_vector(dataframe: pd.DataFrame,
                           metric_col: str = 'Acc_F1_Seed_Mean') -> pd.Series:
    return dataframe.sort_values(by=metric_col, ascending=False).iloc[0]

In [32]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]

In [33]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [34]:
hidden_size_explore = range(40, 61)
lambda_value_explore = np.arange(7.00, 9.00, 0.25)

In [ ]:
hidden_size_result = EvaluationELM.grid_search_hidden_size(X, y, sigmoid, hidden_size_explore)
hidden_size_result = append_unified_metric(hidden_size_result)
best_hidden_size = extract_optimal_vector(hidden_size_result)
best_hidden_size


Hidden Node Size: 40
Hidden Node Size: 41
Hidden Node Size: 42
Hidden Node Size: 43


In [ ]:
lambda_result = EvaluationELM.grid_search_lambda(X, y, sigmoid, best_hidden_size['Hidden_Nodes'], lambda_value_explore)
lambda_result = append_unified_metric(lambda_result)
best_lambda_value = extract_optimal_vector(lambda_result)
best_lambda_value

In [ ]:
combination_result = EvaluationELM.grid_search_hidden_size_and_lambda(X, y, sigmoid, hidden_size_explore, lambda_value_explore)
combination_result = append_unified_metric(combination_result)
best_combination_result = extract_optimal_vector(combination_result)
best_combination_result

In [ ]:
model_configs = [
    {
        "Model": "Phase 1: Best Hidden Nodes",
        "Hidden_Nodes": best_hidden_size['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_hidden_size['Lambda_Value']
    },
    {
        "Model": "Phase 2: Best Lambda (Fixed Node)",
        "Hidden_Nodes": best_lambda_value['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_lambda_value['Lambda_Value']
    },
    {
        "Model": "Phase 3: Best Combined Grid",
        "Hidden_Nodes": best_combination_result['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_combination_result['Lambda_Value']
    }
]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=45,
    stratify=y
)

x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)


main_scaler = StandardScaler()
x_train_scaled = pd.DataFrame(main_scaler.fit_transform(x_train), columns=X.columns)
x_test_scaled  = pd.DataFrame(main_scaler.transform(x_test), columns=X.columns)

features_size = X.shape[1]

In [ ]:
for config in model_configs:
    print(f"=== Executing: {config['Model']} ===")

    elm = ExtremeLearningMachine(
        features_size=features_size,
        hidden_size=config["Hidden_Nodes"],
        activation_function=config["Activation"],
        regularization_lambda=config["Lambda_Value"]
    )

    # Initialize with a fixed seed to ensure your final report is reproducible
    elm.initialize_random_weights(random_seed=45)

    # CRITICAL FIX: Fit and Predict using the SCALED arrays
    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())
    print("\n")